In [3]:
# ==============================================================================
# COLLABORATOR SETUP INSTRUCTIONS (MUST READ)
# ==============================================================================
# If this is NOT your Drive, the code will fail unless you do the following:
#
# 1. Open Google Drive in your browser.
# 2. Go to the "Shared with me" section on the left sidebar.
# 3. Find the folder: "commonvoice-v24_en-AU 3"
# 4. Right-click the folder -> Select "Organize" -> Select "Add shortcut".
# 5. Choose "My Drive" and click "Add".
#
# IMPORTANT: When you run the cell below, a Google login popup will appear.
# You MUST check the box: "See, edit, create, and delete all your Google Drive files."
# If you don't check that box, you will get a "MessageError".

import os

shared_folder_path = "data/commonvoice-v24_en-AU 3"

if os.path.exists(shared_folder_path):
    print(" Accessing the public folder successfully!")

 Accessing the public folder successfully!


In [5]:
# Verify the path of the dataset/
!ls 'data/commonvoice-v24_en-AU 3' | head -n 10

audio_files
commonvoice-v24_en-AU-split.csv
commonvoice-v24_en-AU.csv


In [11]:
import os
import librosa
import soundfile as sf
from tqdm import tqdm

# We use the AUDIO_FILES_DIR defined in the previous setup block
# If you haven't run that block, ensure AUDIO_FILES_DIR is defined
try:
    source_dir = "data/commonvoice-v24_en-AU 3/audio_files"
    target_dir = "data/processed/normalized_au_wavs"
    os.makedirs(target_dir, exist_ok=True)

    files = [f for f in os.listdir(source_dir) if f.endswith('.mp3')]
    print(f"Normalizing {len(files)} files to 16kHz Mono WAV...")

    for file in tqdm(files):
        try:
            # 1. Load MP3 (Librosa automatically resamples to 16k and converts to mono)
            # sr=16000 ensures 16kHz, mono=True is default
            audio, sr = librosa.load(os.path.join(source_dir, file), sr=16000)

            # 2. Peak Normalization (using Librosa's util)
            # This scales the audio so the loudest peak is 1.0 (0 dB) or slightly less
            # To get -1dB, we multiply by roughly 0.89
            peak = max(abs(audio))
            if peak > 0:
                audio = audio * (0.89 / peak)

            # 3. Save as WAV to local Colab storage
            wav_name = file.replace(".mp3", ".wav")
            sf.write(os.path.join(target_dir, wav_name), audio, sr)

        except Exception as e:
            print(f"Skipping {file}: {e}")

    print(f"\n Normalization complete! {len(os.listdir(target_dir))} files are in {target_dir}")

except NameError:
    print(" Error: AUDIO_FILES_DIR is not defined. Please run the Setup Block first!")

Normalizing 3952 files to 16kHz Mono WAV...


100%|██████████| 3952/3952 [00:26<00:00, 149.40it/s]


 Normalization complete! 3952 files are in data/processed/normalized_au_wavs


In [21]:
# 3. CREATE SPEAKER-AWARE MANIFESTS
print("\n📝 Creating Train/Test splits...")
df = pd.read_csv(csv_path)

# Map clips to speakers to prevent data leakage
speaker_to_clips = defaultdict(list)
for _, row in df.iterrows():
    wav_filename = row['path'].replace('.mp3', '.wav')
    wav_path = os.path.join(target_dir, wav_filename)

    if os.path.exists(wav_path):
        speaker_to_clips[row['client_id']].append({
            "wav": wav_path,
            "duration": float(row['duration_ms']) / 1000.0,
            "spk_id": str(row['client_id'])
        })

# Split Speakers (80% Train, 20% Test)
all_speakers = list(speaker_to_clips.keys())
random.seed(2026)
random.shuffle(all_speakers)

split_idx = int(len(all_speakers) * 0.8)
train_spks = all_speakers[:split_idx]
test_spks = all_speakers[split_idx:]

# Helper to save JSON
def save_sub_manifest(speaker_list, filename):
    sub_manifest = {}
    for spk in speaker_list:
        for i, clip in enumerate(speaker_to_clips[spk]):
            # Create a unique ID for each utterance
            utt_id = f"{spk}_{i}"
            sub_manifest[utt_id] = clip
    with open(filename, 'w') as f:
        json.dump(sub_manifest, f, indent=4)
    return len(sub_manifest)

train_count = save_sub_manifest(train_spks, 'train.json')
test_count = save_sub_manifest(test_spks, 'test.json')

print(f" Normalization & Splitting Complete!")
print(f" Normalized files: {len(os.listdir(target_dir))}")
print(f" train.json: {train_count} clips ({len(train_spks)} speakers)")
print(f" test.json: {test_count} clips ({len(test_spks)} speakers)")


📝 Creating Train/Test splits...
✅ Normalization & Splitting Complete!
📂 Normalized files: 3952
📄 train.json: 3069 clips (334 speakers)
📄 test.json: 882 clips (84 speakers)


In [26]:
# CONVERT MASTER MANIFEST INTO TRAIN/TEST SPLITS

import json
import random
import os
from collections import defaultdict

# 1. Load both existing files
files_to_merge = ['train.json', 'test.json']
master_data = {}

print("🔄 Merging existing files to prepare for speaker-aware split...")

for filename in files_to_merge:
    if os.path.exists(filename):
        with open(filename, 'r') as f:
            data = json.load(f)
            master_data.update(data)
            print(f" Loaded {len(data)} clips from {filename}")
    else:
        print(f" Warning: {filename} not found. Skipping...")

if not master_data:
    print(" Error: No data found. Please check your file names in the sidebar!")
else:
    # 2. Group all audio clips by Speaker ID (spk_id)
    speaker_to_clips = defaultdict(list)
    for clip_id, metadata in master_data.items():
        speaker_to_clips[metadata['spk_id']].append((clip_id, metadata))

    all_speakers = list(speaker_to_clips.keys())
    random.seed(2026)
    random.shuffle(all_speakers)

    # 3. Calculate split points (80% Train / 20% Test)
    total_spks = len(all_speakers)
    split_point = int(total_spks * 0.8)

    train_spks = set(all_speakers[:split_point])
    test_spks = set(all_speakers[split_point:])

    # 4. Helper function to create subsets
    def create_subset(speaker_set):
        subset = {}
        for spk in speaker_set:
            for clip_id, metadata in speaker_to_clips[spk]:
                subset[clip_id] = metadata
        return subset

    train_data = create_subset(train_spks)
    test_data = create_subset(test_spks)

    # 5. Overwrite the files with the new, clean split
    for name, data, spk_set in [("train", train_data, train_spks), ("test", test_data, test_spks)]:
        filename = f"{name}.json"
        with open(filename, 'w') as f:
            json.dump(data, f, indent=4)
        print(f"📂 Re-saved {filename:10} | {len(data):5} clips | {len(spk_set):4} speakers")

    # 6. Leakage Verification
    overlap = train_spks.intersection(test_spks)
    if not overlap:
        print("-" * 45)
        print(" SUCCESS: Files are now synchronized and speaker-isolated!")
    else:
        print(f" WARNING: Leakage detected among {len(overlap)} speakers.")

🔄 Merging existing files to prepare for speaker-aware split...
✅ Loaded 3069 clips from train.json
✅ Loaded 882 clips from test.json
📂 Re-saved train.json |  2484 clips |  334 speakers
📂 Re-saved test.json  |  1467 clips |   84 speakers
---------------------------------------------
🚀 SUCCESS: Files are now synchronized and speaker-isolated!
